# Detecting Missing Values in DataFrames

This notebook demonstrates how to identify and summarize missing values in Pandas DataFrames. Understanding missing data is critical for data quality assessment and planning the next steps in analysis.

## Part 1: Loading DataFrames with Missing Values

We'll work with datasets that contain intentional missing values (represented as empty cells in CSV files).

In [ ]:
import pandas as pd
import numpy as np

# Load datasets with missing values
employees_df = pd.read_csv('../employees_with_missing.csv')
sales_df = pd.read_csv('../sales_with_missing.csv')

print("="*60)
print("EMPLOYEES DATAFRAME WITH MISSING VALUES")
print("="*60)
print(employees_df)

print("\n" + "="*60)
print("SALES DATAFRAME WITH MISSING VALUES")
print("="*60)
print(sales_df)

## Part 2: Detecting Missing Values

Pandas represents missing values as `NaN` (Not a Number) or `None`. We can identify them using several methods.

### 2.1: Using `.isna()` and `.isnull()` - Element-Level Detection

These methods return a Boolean DataFrame showing `True` for missing values and `False` for present values.

In [ ]:
# .isna() and .isnull() are equivalent - both identify missing values
missing_mask_employees = employees_df.isna()

print("Missing Value Boolean Mask (Employees):")
print("(True = Missing, False = Present)")
print(missing_mask_employees)

print("\n" + "="*60 + "\n")

# Show this more clearly for one column
print("Missing Values in Department Column Only:")
print(missing_mask_employees['Department'])
print("\nType:", type(missing_mask_employees['Department']))

### 2.2: Using `.notna()` - Inverse Detection

`.notna()` is the opposite of `.isna()` - it shows `True` for present values.

In [ ]:
# .notna() shows where data IS present
present_mask = employees_df.notna()

print("Present Value Boolean Mask (Employees):")
print("(True = Present, False = Missing)")
print(present_mask)

print("\n" + "="*60 + "\n")

print("Employees with NO missing values:")
rows_with_all_data = employees_df[employees_df.notna().all(axis=1)]
print(rows_with_all_data)

## Part 3: Summarizing Missing Values by Column

This is one of the most important analysis steps - understanding which columns have missing data and how much.

### 3.1: Count Missing Values Per Column

Let's see how many missing values exist in each column.

In [ ]:
# Count missing values per column
missing_counts = employees_df.isna().sum()

print("Missing Value Counts per Column (Employees):")
print(missing_counts)

print("\n" + "="*60 + "\n")

# More detailed view with percentage
print("Missing Value Summary (Employees):")
total_rows = len(employees_df)
for column in employees_df.columns:
    missing_count = employees_df[column].isna().sum()
    missing_pct = (missing_count / total_rows) * 100
    print(f"{column:15} | Missing: {missing_count:2} | Percentage: {missing_pct:5.1f}%")

### 3.2: Missing Data Overview with `.info()`

The `.info()` method provides a comprehensive view of the DataFrame, including missing value information.

In [ ]:
print("DataFrame Info - Employees:")
employees_df.info()

print("\n" + "="*60 + "\n")

print("DataFrame Info - Sales:")
sales_df.info()

### 3.3: Missing Data Overview with `.describe()`

`.describe()` shows statistics for numeric columns, but notice it doesn't count NaN values by default.

In [ ]:
print("Summary Statistics (Employees) - Note: NaN are excluded from counts")
print(employees_df.describe())

print("\n" + "="*60 + "\n")

# Better: include count explicitly
print("Summary with NaN Counts (Employees):")
summary = employees_df.describe().T
summary['missing'] = employees_df.isna().sum()
summary['total'] = len(employees_df)
summary['missing_pct'] = (summary['missing'] / summary['total'] * 100).round(1)
print(summary[['count', 'missing', 'missing_pct']])

## Part 4: Inspecting Rows Containing Missing Values

Now let's identify and examine which rows have missing data.

### 4.1: Finding Rows with ANY Missing Values

In [ ]:
# Find rows with at least one missing value
has_missing = employees_df.isna().any(axis=1)

print("Rows with At Least One Missing Value:")
print(has_missing)

print("\n" + "="*60 + "\n")

rows_with_missing = employees_df[has_missing]
print(f"Found {len(rows_with_missing)} rows with missing values:")
print(rows_with_missing)

### 4.2: Finding Rows Missing Specific Columns

Let's identify rows missing specific critical columns.

In [ ]:
# Rows missing Department
missing_department = employees_df[employees_df['Department'].isna()]
print("Rows Missing Department:")
print(missing_department)

print("\n" + "="*60 + "\n")

# Rows missing Salary
missing_salary = employees_df[employees_df['Salary'].isna()]
print("Rows Missing Salary:")
print(missing_salary)

print("\n" + "="*60 + "\n")

# Rows missing EITHER Department OR Salary
missing_key_fields = employees_df[
    employees_df['Department'].isna() | employees_df['Salary'].isna()
]
print("Rows Missing Department OR Salary (Critical Fields):")
print(missing_key_fields)

### 4.3: Finding Completely Empty Rows

Some rows might be entirely empty (all values missing).

In [ ]:
# Rows where ALL values are missing
completely_empty = employees_df[employees_df.isna().all(axis=1)]
print(f"Completely Empty Rows: {len(completely_empty)}")
if len(completely_empty) > 0:
    print(completely_empty)
else:
    print("(None found)")

print("\n" + "="*60 + "\n")

# Rows where MOST values are missing (e.g., > 50%)
mostly_empty = employees_df[
    (employees_df.isna().sum(axis=1) / len(employees_df.columns)) > 0.5
]
print(f"Rows with >50% Missing Data: {len(mostly_empty)}")
if len(mostly_empty) > 0:
    print(mostly_empty)
else:
    print("(None found)")

## Part 5: Visual Summary of Missing Data

Creating a clear overview of missing value patterns across the entire dataset.

In [ ]:
# Create a comprehensive missing data summary
print("="*70)
print("COMPREHENSIVE MISSING DATA SUMMARY - EMPLOYEES")
print("="*70)

missing_summary = pd.DataFrame({
    'Missing_Count': employees_df.isna().sum(),
    'Missing_Percentage': (employees_df.isna().sum() / len(employees_df) * 100).round(1),
    'Present_Count': employees_df.notna().sum(),
    'Data_Type': employees_df.dtypes
})

print(missing_summary)

print("\n" + "="*70)
print("COMPREHENSIVE MISSING DATA SUMMARY - SALES")
print("="*70)

missing_summary_sales = pd.DataFrame({
    'Missing_Count': sales_df.isna().sum(),
    'Missing_Percentage': (sales_df.isna().sum() / len(sales_df) * 100).round(1),
    'Present_Count': sales_df.notna().sum(),
    'Data_Type': sales_df.dtypes
})

print(missing_summary_sales)

## Part 6: Understanding Impact of Missing Values

Missing values can affect analysis in different ways. Let's demonstrate the impact.

### 6.1: Impact on Calculations

When performing calculations, Pandas automatically excludes NaN values. This can lead to misleading results if not verified.

In [ ]:
print("IMPACT ON CALCULATIONS")
print("="*70)

# Calculate average salary
avg_salary_all = employees_df['Salary'].mean()
non_missing_salary = employees_df['Salary'].notna().sum()
missing_salary_count = employees_df['Salary'].isna().sum()

print(f"Average Salary: ${avg_salary_all:,.2f}")
print(f"Calculated from: {non_missing_salary} employees (out of {len(employees_df)})")
print(f"Missing Salary Values: {missing_salary_count}")
print(f"\nIMPACT: Analysis excluded {missing_salary_count} employees from salary calculation!")

print("\n" + "="*70 + "\n")

# Show which employees are EXCLUDED from the calculation
excluded_from_salary = employees_df[employees_df['Salary'].isna()][['EmployeeID', 'Name', 'Department']]
print("Employees EXCLUDED from Salary Average:")
print(excluded_from_salary)

print("\n" + "="*70 + "\n")

# Sales data: impact on total revenue
total_revenue = sales_df['Total'].sum()
non_missing_total = sales_df['Total'].notna().sum()
missing_total_count = sales_df['Total'].isna().sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Calculated from: {non_missing_total} transactions (out of {len(sales_df)})")
print(f"Missing Total Values: {missing_total_count}")
print(f"\nIMPACT: Revenue calculation EXCLUDES {missing_total_count} unknown transactions!")

### 6.2: Impact on Grouping and Filtering

Missing values in grouping columns can cause data loss.

In [ ]:
print("IMPACT ON GROUPING")
print("="*70)

# Group by Department
print("Salary by Department:")
dept_salary = employees_df.groupby('Department')['Salary'].agg(['count', 'mean', 'sum'])
print(dept_salary)

print("\n" + "="*70 + "\n")

# Show missing department values
missing_dept = employees_df[employees_df['Department'].isna()][['EmployeeID', 'Name', 'Salary']]
print("Employees with Missing Department (NOT INCLUDED in grouping above):")
print(missing_dept)

print(f"\nIMPACT: {len(missing_dept)} employees are EXCLUDED from department analysis!")

## Part 7: Patterns in Missing Data

Let's identify if missing values follow any patterns (which could indicate systematic problems).

In [ ]:
print("MISSING DATA PATTERNS")
print("="*70)

# Check if rows that are missing one value tend to be missing others
print("Missing Value Co-occurrence (Employees):")
missing_corr = employees_df.isna().corr()
print(missing_corr)

print("\n" + "="*70 + "\n")

# Show rows with multiple missing values
print("Rows with Multiple Missing Values:")
missing_per_row = employees_df.isna().sum(axis=1)
for idx, row in employees_df.iterrows():
    m_count = missing_per_row[idx]
    if m_count > 1:
        print(f"\nRow {idx}: {m_count} missing values")
        missing_columns = employees_df.columns[employees_df.iloc[idx].isna()].tolist()
        print(f"  Missing columns: {missing_columns}")

## Part 8: Practical Steps Before Continuing Analysis

This section outlines what you should do upon discovering missing values.

In [ ]:
print("CHECKLIST: What to Do When You Discover Missing Values")
print("="*70)

# Step 1: Document the extent
print("\n1. DOCUMENT EXTENT OF MISSING DATA")
print("   ✓ Check which columns have missing values")
for col in employees_df.columns:
    if employees_df[col].isna().any():
        count = employees_df[col].isna().sum()
        pct = (count / len(employees_df) * 100)
        print(f"     - {col}: {count} missing ({pct:.1f}%)")

# Step 2: Understand the impact
print("\n2. UNDERSTAND IMPACT ON YOUR ANALYSIS")
print("   ✓ Identify which analyses are AFFECTED by missing values")
print("     - Salary analysis: AFFECTED (2 missing values)")
print("     - Department grouping: AFFECTED (2 missing values)")
print("     - Phone directory: AFFECTED (2 missing values)")

# Step 3: Decide on handling strategy
print("\n3. DECIDE ON HANDLING STRATEGY")
print("   Options for each column:")
for col in employees_df.columns:
    missing_count = employees_df[col].isna().sum()
    if missing_count > 0:
        pct = (missing_count / len(employees_df) * 100)
        print(f"   - {col}: {missing_count} missing values ({pct:.1f}%)")
        if pct < 5:
            print(f"       → Low percentage: Could DROP these rows")
        elif pct < 20:
            print(f"       → Moderate percentage: Could DROP or IMPUTE")
        else:
            print(f"       → High percentage: Should IMPUTE or EXCLUDE this column")

# Step 4: Document decisions
print("\n4. DOCUMENT YOUR DECISION")
print("   ✓ Example decisions for this dataset:")
print("     - Salary & Department: Drop rows (they're critical)")
print("     - PhoneNumber: Drop column (optional field, too many missing)")
print("     - HireDate: Impute with 'Unknown' or drop rows")

# Step 5: Verify before proceeding
print("\n5. VERIFY BEFORE CONTINUING")
print("   ✓ After handling missing values:")
print("     - Check for new missing values introduced")
print("     - Verify row/column counts are as expected")
print("     - Sample the cleaned data to spot-check results")

## Part 9: Common Pitfall - Proceeding Without Verification

This section demonstrates the consequences of ignoring missing values.

In [ ]:
print("SCENARIO: Analysis Gone Wrong Due to Missing Values")
print("="*70)

# PITFALL: Assuming all data is present
print("\n❌ PITFALL: Making assumptions without checking for missing values\n")

# Example 1: Wrong count calculations
assuming_no_missing = len(employees_df)
actual_employees_with_salary = employees_df['Salary'].notna().sum()

print(f"Assumption: We have {assuming_no_missing} employees")
print(f"Reality: Only {actual_employees_with_salary} have salary data")
print(f"ERROR: {assuming_no_missing - actual_employees_with_salary} employees' data was silently excluded!")

print("\n" + "="*70 + "\n")

# Example 2: Incorrect grouping statistics
print("❌ PITFALL: Reporting percentages based on incomplete data\n")

depts = employees_df['Department'].unique()
print(f"Department distribution (INCOMPLETE):")
for dept in depts:
    if pd.notna(dept):
        count = len(employees_df[employees_df['Department'] == dept])
        pct = (count / len(employees_df) * 100)
        print(f"  {dept}: {count} employees ({pct:.1f}%)")

missing_dept_count = employees_df['Department'].isna().sum()
print(f"  (Unknown): {missing_dept_count} employees ({missing_dept_count/len(employees_df)*100:.1f}%)")

engineering_pct = (len(employees_df[employees_df['Department'] == 'Engineering']) / len(employees_df) * 100)
print(f"\nProblem: You exclude missing dept data from reporting!")
print(f"         True percentage for Engineering is {engineering_pct:.1f}% (not shown above)")

print("\n" + "="*70 + "\n")

# SOLUTION: Early detection and verification
print("✅ SOLUTION: Check for missing values FIRST\n")

print("Step 1: DETECT missing values immediately after loading data")
print(f"  → {employees_df.isna().sum().sum()} total missing values found")

print("\nStep 2: SUMMARIZE the impact")
print("  → Missing values by column:")
missing_data = pd.DataFrame({
    'Column': employees_df.columns,
    'Missing_Count': employees_df.isna().sum(),
    'Missing_Percentage': (employees_df.isna().sum() / len(employees_df) * 100).round(1)
})
print(missing_data[missing_data['Missing_Count'] > 0])

print("\nStep 3: INSPECT affected rows")
affected_rows = employees_df[employees_df.isna().any(axis=1)]
print(f"  → {len(affected_rows)} rows have at least one missing value")

print("\nStep 4: PLAN handling strategy BEFORE analysis")
print("  → Document which analyses will be affected")
print("  → Choose appropriate method: drop, impute, or exclude column")
print("  → Proceed only after verification")

print("\n✅ Result: Clean, reliable analysis with no surprises!")

## Summary: Key Takeaways

### Detection Methods
- **`.isna()` / `.isnull()`**: Returns Boolean DataFrame showing missing values
- **`.notna()`**: Returns Boolean DataFrame showing present values
- **`.info()`**: Shows missing value counts alongside data types

### Summarizing Missing Data
- **`.isna().sum()`**: Count missing values per column
- **`.isna().mean()`**: Get percentage of missing values
- Use combinations with groupby for more detailed analysis

### Inspecting Affected Rows
- **`.isna().any(axis=1)`**: Find rows with ANY missing values
- **Conditional filtering**: Target specific columns with missing data
- **Co-occurrence analysis**: Identify patterns in missing data

### Impact on Analysis
- Calculations exclude NaN automatically (can cause surprises)
- Grouping operations lose data with missing group keys
- Counts and percentages become inaccurate
- Statistical results may be misleading

### Best Practices
1. **Check immediately after loading**: Don't wait until problems appear
2. **Quantify the impact**: Understand how many rows/columns are affected
3. **Inspect affected data**: See what's missing and why
4. **Plan your strategy**: Decide how to handle missing values
5. **Verify before proceeding**: Never assume data is complete
6. **Document your decisions**: Record what you did and why